
# A1. Model parameterization

Dendra lets you optimize model parameters while still exposing clean, simulation-ready values at the model surface. In practice, that means you can work with attributes such as `model.celsius`, `model.cm`, or `model.rhoa` as the values the model will actually use during `initialize()` and simulation, while Dendra handles the bookkeeping needed for trainable sources, positivity constraints, and region-specific overrides.

This page focuses on the user-facing mental model:

- when to use `GLOBAL`, `BATCH`, `RANGE`, and `PARAMETER`
- how to think about exposed values such as `model.rhoa`
- why `initialize()` is the moment when those values are refreshed
- how to automate relationships with parametrizations and hooks



## Parameter types at a glance

| Declaration | User-facing meaning | Effective shape | Typical use |
| --- | --- | --- | --- |
| `GLOBAL(...)` | One value shared across the whole model | Scalar-like tensor | Temperature, a global scale factor, a shared offset |
| `BATCH(...)` | One value per batch element | Scalar-like tensor | Batch-specific parameters, e.g., random distribution of conductances over the population |
| `RANGE(...)` | One value per compartment (and per axon in a population) | `shape_p` | Axial resistivity, capacitance, compartment-wise conductance |
| `PARAMETER(...)` | A generic learnable object used by your own logic | Whatever you provide | Helper coefficients, custom transforms, grouped coefficients |
| `GLOBALP(...)`, `BATCHP(...)`, `RANGEP(...)`, `PARAMETERP(...)`, `GLOBALN(...)`, `BATCHN(...)`, `RANGEN(...)`, `PARAMETERN(...)` | Same as above, but constrained to be positive / negative | Same as the corresponding non-positive form | Positive-only or negative-only physical quantities (e.g., `GLOBALP(gnabar=0.12)`, `GLOBALN(el=-54.3)`). Useful during optimization. |

Two practical rules matter most:

1. **Use `GLOBAL` when the model should consume a single shared value.**
2. **Use `BATCH` when the model should consume a single value per batch element (cell in the population).**
2. **Use `RANGE` when the model should consume a full compartment-wise tensor.**

A `RANGE` parameter does **not** force every compartment to be different. It simply means the model *can* hold a different value at every compartment. A scalar default such as `RANGE(rhoa=35.4)` is still valid; Dendra materializes it as a full tensor over compartments.



## The key mental model

For everyday use, it is helpful to think in terms of **effective values**:

- `model.celsius` is the temperature that `initialize()` and the mechanisms see.
- `model.rhoa` is the compartment-wise axial resistivity tensor that the model actually uses.
- `model.cm` is the compartment-wise capacitance tensor that the model actually uses.

That is why user code usually reads naturally:

```python
model.celsius
model.rhoa
model.cm
```

The important distinction is that `GLOBAL` and `RANGE` expose **simulation-ready values**, while Dendra keeps the underlying trainable sources and bookkeeping behind the scenes. This is what makes it possible to:

- refresh values during `initialize()`
- impose positivity / negativity constraints with `GLOBALP` / `BATCHP` / `RANGEP`, etc.
- define region-specific overrides for a `RANGE` variable
- register derived relationships without manually editing leaf tensors

> **Note**
> For `PARAMETERP(...)`, the attribute itself is a positive parameter object, so use `self.evaluate('name')` when you need the numeric value. For `GLOBALP(...)` and `RANGEP(...)`, the model attribute is already the usable positive value, so `self.name` gives you the numeric value directly.



## `RANGE` means "per compartment"

This is the most important user-facing point to internalize:

- A `GLOBAL` parameter gives the model **one shared value**.
- A `RANGE` parameter gives the model **one value for every compartment**.

That compartment-wise tensor can be used in several styles:

1. **Uniform**: every compartment gets the same value.
2. **Region-wise**: several compartments share a value because they belong to the same region.
3. **Fully compartment-specific**: every compartment can take a different value.

The figure below shows region-wise sharing (above) and a fully compartment-specific range tensor (below).


In [ ]:
from range_viz import render, render2; render()


## Real examples in Dendra models

In the core axon models, you will see declarations such as:

```python
P.RANGE(cm=1.0, rhoa=35.4)
P.GLOBAL(celsius=37.0)
```

That gives the following user-facing behavior:

```python
model.celsius        # one shared value
model.cm             # one value per compartment
model.rhoa           # one value per compartment
model.rhoa.shape     # matches the model's compartment shape
```

So even though `rhoa` might *start* from a scalar default like `35.4`, the model-facing value is still a full compartment-wise tensor.



## Region-specific parameterization of a `RANGE` variable

A very common pattern is to start from one baseline `RANGE` value and then override selected compartments with additional learned sources.

For example, suppose you want one baseline `rhoa` everywhere, but different values for nodes, internodes, and terminals:

```python
# Example sketch: keys identify subsets of compartments
node_key = ...
internode_key = ...
terminal_key = ...

# Baseline RANGE source already exists because rhoa was declared with RANGE(...)
# Add three more region-specific sources
model.parametrize("rhoa", 70.0, key=node_key)        # creates rhoa_param_0
model.parametrize("rhoa", 40.0, key=internode_key)   # creates rhoa_param_1
model.parametrize("rhoa", 55.0, key=terminal_key)    # creates rhoa_param_2

# equivalently, parameterize via Slice objects
model[node_key].parametrize("rhoa", 70.0)
model[internode_key].parametrize("rhoa", 40.0)
model[terminal_key].parametrize("rhoa", 55.0)

# parameters associated with mechanisms can be set during insertion
model[node_key].insert(hh, gnabar=0.5)
model[internode_key].insert(hh, gnabar=0.01)
model[terminal_key].insert(hh, gnabar=0.0)

# or after building, since the parameterization system is flexible and dynamic
# note that in this case, the 'hh' mechanism must already be present at the node_key compartments, e.g., by calling model.insert(hh) before
model.insert(hh)  # ensure hh is present everywhere
model.build()
model[node_key].mech.hh.parametrize("gnabar", 0.5)

# it is also possible to use a torch.nn.Parameter or equivalent (e.g., Dendra's own PositiveParam), if you want e.g., different regions to be governed by the same underlying parameter object
gnabar_param = torch.nn.Parameter(torch.tensor(0.5))
model[node_key].mech.hh.parametrize("gnabar", gnabar_param)
model[internode_key].mech.hh.parametrize("gnabar", gnabar_param)

# After all parameterizations are in place, call initialize() to materialize the final compartment-wise tensor seen by the model

model.initialize()
```

After `initialize()`, `model.rhoa` contains the **final compartment tensor** seen by the model.

Two useful details:

- If you call `parametrize(...)` without an alias, Dendra creates names such as `rhoa_param_0`, `rhoa_param_1`, and so on.
- If you pass `alias="node"`, `alias="internode"`, etc., the added sources become easier to inspect by name.

This is the key reason `RANGE` is so flexible: the model consumes a compartment-wise tensor, but you are free to parameterize that tensor with one source, a few region-wise sources, or a fully compartment-specific construction.



## What `initialize()` does in practice

From a user's point of view, `initialize()` is the moment when Dendra refreshes the effective parameter values that the model will actually consume.

A good working summary is:

1. build the model graph if needed
2. refresh exposed `GLOBAL`, `BATCH`, and `RANGE` values from their sources
3. apply region-specific overrides and registered parametrizations
4. initialize voltage / mechanisms
5. run any pre- and post-initialize hooks you registered

That is why, after changing a parameterization, the usual next step is to call:

```python
model.initialize()
```

The two diagrams below show the flow for one `GLOBAL` variable and one `RANGE` variable.



### Call graph: `initialize()` for a `GLOBAL` variable (`celsius`)

Here the important point is that one hidden source gives rise to one shared effective value, and that effective value is what the mechanism initialization reads.


In [ ]:
from IPython.display import Image, display
display(Image(filename='global_param.png'))


### Call graph: `initialize()` for a `RANGE` variable (`rhoa`) with region-specific sources

This example shows a baseline `rhoa` source plus three extra sources that control different subsets of compartments. The model still consumes a single final tensor, `model.rhoa`, but that tensor can be assembled from multiple underlying learned values.


In [ ]:
display(Image(filename='range_param.png'))

In [ ]:
render2()


## Automating relationships without manual tensor surgery

Dendra gives you three user-facing automation patterns, each useful for a slightly different job.

### 1. Parametrizations for differentiable relationships

Use a parametrization when one model quantity should be computed from another quantity or from learned coefficients.

Examples from the built-in models include:

- deriving node diameter from fiber diameter
- scaling `rhoa` from geometric quantities in the myelinated model

This keeps the transformation attached to the model logic rather than forcing you to edit a learnable tensor in place.

### 2. In-graph parametrizations for exposed `GLOBAL` / `RANGE` values

When you want the **exposed value** (for example `model.rhoa`) to be transformed automatically during buffer refresh, register a parametrization on that value instead of editing the underlying learnable source manually.

This is especially helpful when the model should consume a derived tensor, but you still want the trainable sources to stay clean and stable.

### 3. Pre/post initialize hooks for setup logic

Hooks are ideal for initialization-time automation:

- set or reset buffers/state each time `initialize()` runs
- apply convenience setup logic around mechanism initialization
- keep repetitive setup code out of your experiment script

A lightweight example is `set_value(...)`, which registers a post-initialize action for a named variable.

```python
# Example pattern
model.set_value("v", torch.full_like(model.v, -80.0))
model.initialize()
```

If you need custom logic, you can register your own hook:

```python
def my_setup(model):
    # operate on the exposed model values or states here
    model.rhoa[..., 0] = model.rhoa[..., 0] * 1.1

model.register_post_initialize_hook(my_setup)
model.initialize()
```

### Which one should you choose?

- **Use a parametrization** when the relationship should be part of the model's learnable transformation pipeline.
- **Use a hook** when you want initialization-time setup or convenience logic.
- In both cases, the main benefit is the same: **you avoid manually rewriting the leaf trainable tensors that Dendra is optimizing.**



## Random parameters and runtime noise

Dendra has two related stochastic declaration families.

### Quenched random parameters: `GLOBALRAND`, `BATCHRAND`, `RANGERAND`

Use `RAND` declarations for heterogeneity that is sampled during initialization and then held fixed during the simulation:

```python
from dendra.models.mechanisms import Mechanism

class NoisyLeak(Mechanism):
    Mechanism.RANGERAND(
        "g_scale",
        distribution="truncated_normal",
        mu=1.0,
        sigma=0.10,
        low=0.5,
        high=1.5,
    )
```

This creates a sampled buffer named `g_scale` plus ordinary Dendra distribution-parameter buffers:

```text
g_scale_mu
g_scale_sigma
g_scale_low
g_scale_high
```

The distribution parameters can be overridden locally at insertion time just like normal `RANGE` parameters:

```python
cell.soma.insert(NoisyLeak, alias="soma", g_scale_sigma=0.02)
cell.dend.insert(NoisyLeak, alias="dend", g_scale_sigma=0.15)
```

By default, `RAND` samples are re-generated on each top-level `initialize()` call. You can explicitly resample them with:

```python
cell.resample_random_parameters("g_scale")
```

`RAND` samples use the reparameterized path by default for continuous distributions, so gradients can flow to distribution parameters such as `g_scale_mu` and `g_scale_sigma` when the rest of the computation graph supports it.

### Runtime noise: `GLOBALNOISE`, `BATCHNOISE`, `RANGENOISE`

Use `NOISE` declarations for exogenous stochastic drives that are resampled during simulation. For v1, runtime noise is sampled before mechanism state advancement in each timestep and is written into a stable buffer in-place under `torch.no_grad()`:

```python
from dendra.models.mechanisms import Mechanism

class NoisyCurrent(Mechanism):
    Mechanism.RANGENOISE(
        "eta",
        distribution="normal",
        mu=0.0,
        sigma=1.0,
    )
    Mechanism.RANGE(sigma_i=0.01)

    def i(self, v):
        return self.sigma_i * self.eta
```

This is the fast simulation path. Gradients do **not** flow through the runtime-noise distribution parameters. For differentiable stochastic dynamics in `State` variables, use the SDE support described in the mechanism/state authoring guide.

You can manually refresh runtime noise outside the normal simulation loop with:

```python
cell.resample_runtime_noise("eta")
```

Runtime-noise distribution parameters are still ordinary Dendra parameters, so local overrides work:

```python
cell.apic.insert(NoisyCurrent, alias="apical", eta_sigma=0.5)
cell.soma.insert(NoisyCurrent, alias="soma", eta_sigma=0.1)
```

The first v1 runtime-noise cadence and phase are `cadence="step"` and `phase="pre_state"`, meaning the buffer is refreshed once per mechanism advance.

### Built-in and custom distributions

Built-in distributions include:

```text
normal(mu, sigma)
lognormal(mu, sigma)
uniform(low, high)
truncated_normal(mu, sigma, low, high)
```

You can register a custom distribution globally:

```python
def constant_sampler(parameters, rng, shape, *, device, dtype):
    return torch.empty(shape, device=device, dtype=dtype).fill_(parameters["value"])

dn.register_random_distribution(
    "constant",
    {"value": 1.0},
    sampler=constant_sampler,
)
```



## Takeaways

- `GLOBAL` means **one effective value shared across the model**.
- `BATCH` means **one effective value per batch element (cell in the population)**.
- `RANGE` means **one effective value per compartment**.
- A `RANGE` parameter can be uniform, region-wise, or fully compartment-specific.
- `model.rhoa` is best understood as the **final compartment tensor the model uses**, not just a raw leaf parameter.
- Re-running `initialize()` refreshes those effective values after you change parameterization.
- Parametrizations and initialize hooks are the preferred way to automate transformations while keeping the underlying trainable sources well behaved.

- `RAND` declarations represent initialization-time heterogeneity; `NOISE` declarations represent detached runtime stochastic drives.
- `NOISE` variables are optimized for simulation and do not preserve gradients through their distribution parameters.